# Federated (GradAvg) — Answer-Contrastive Objective (ORPO)

$$\mathcal{L} = \mathcal{L}_{\mathrm{SFT}} + \lambda_y \cdot \left[-\log \sigma\left(\log\mathrm{odds}_{\theta}(y^{+} \mid q, c^{+}) - \log\mathrm{odds}_{\theta}(y^{-} \mid q, c^{+})\right)\right]$$

Supervised loss plus an ORPO odds-ratio term contrasting the **preferred against
the general answer** under the *same* relevant snippet. Odds are formed from
length-normalised log-likelihoods. `ORPO_LAMBDA = 0.2`.

---

*Notation:* `q` query · `c+` relevant snippet · `c-` off-topic snippet from the same user · `y+` personalised answer · `y-` general answer


In [1]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip -q install -U datasets huggingface_hub transformers accelerate peft bitsandbytes rouge-score

    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path(".")

if IN_COLAB:
    PP_ROOT = DRIVE_ROOT / "privacy_perserving_pllm"
else:
    _here = Path(".").resolve()
    PP_ROOT = next(
        (
            p for p in [_here, *_here.parents]
            if (p / "v2_personamem_persona_subsets").exists()
            or all((p / d).exists() for d in ("centralised", "evaluation", "fed_grad_avg", "zero_shot"))
        ),
        _here,
    )
SUBSET_NAME = "all_subset"
SUBSETS_DIR = PP_ROOT / "v2_personamem_persona_subsets"

RESULTS_DIR = PP_ROOT / SUBSET_NAME / "v2_personamem_federated_orpo_gradavg_snippet"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Subset:", SUBSET_NAME)
print("Results will be saved to:", RESULTS_DIR.resolve())

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 236.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 59.5 MB/s eta 0:00:00
Mounted at /content/drive
Subset: all_subset
Results will be saved to: /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_federated_orpo_gradavg_snippet


In [2]:
import ast
import json
import random
from typing import Any, Dict, List

from tqdm.auto import tqdm

import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

DATASET_NAME = "bowen-upenn/PersonaMem-v2"
CONFIG = "benchmark"
TEXT_SPLITS = ("train_text", "val_text", "benchmark_text")

SEED = 42
TOKENIZER_NAME = "Qwen/Qwen3-0.6B"
random.seed(SEED)


def load_split(split):
    if split not in TEXT_SPLITS:
        raise ValueError(f"split must be one of {TEXT_SPLITS}, got {split!r}")
    return load_dataset(DATASET_NAME, CONFIG, split=split)


def parse_user_query(raw):
    if isinstance(raw, dict):
        return str(raw.get("content", raw)).strip()
    if isinstance(raw, str):
        try:
            d = ast.literal_eval(raw)
            if isinstance(d, dict):
                return str(d.get("content", raw)).strip()
        except (ValueError, SyntaxError):
            pass
        return raw.strip()
    return str(raw).strip()


def parse_incorrect_answers(raw):
    if isinstance(raw, list):
        return [str(x) for x in raw]
    if hasattr(raw, "tolist"):
        return [str(x) for x in raw.tolist()]
    if isinstance(raw, str):
        try:
            val = ast.literal_eval(raw)
            if isinstance(val, list):
                return [str(x) for x in val]
        except (ValueError, SyntaxError):
            pass
        return [raw]
    return []

def get_snippet(row):
    val = row.get("related_conversation_snippet")
    if val is None:
        return ""
    return str(val).strip()

In [3]:
def load_subset(name, subsets_dir=SUBSETS_DIR):
    """Load a persona subset (ids + train/val rows) saved by create_persona_subsets.ipynb."""
    subset_dir = Path(subsets_dir) / name
    if not subset_dir.exists():
        raise FileNotFoundError(f"Subset not found: {subset_dir}. Run create_persona_subsets.ipynb first.")
    with open(subset_dir / "personas.json", "r", encoding="utf-8") as f:
        persona_ids = [int(p) for p in json.load(f)]
    tr = pd.read_parquet(subset_dir / "train.parquet")
    va = pd.read_parquet(subset_dir / "val.parquet")
    return sorted(persona_ids), tr, va

MIN_VAL_ROWS = 4
CLIENT_PERSONAS, train_df, val_df = load_subset(SUBSET_NAME)
NUM_CLIENTS = len(CLIENT_PERSONAS)
val_counts = val_df.groupby("persona_id").size()
print(f"train rows: {len(train_df):,} | val rows: {len(val_df):,}")
print(f"Subset '{SUBSET_NAME}': {NUM_CLIENTS} personas")
print(f"Client personas ({len(CLIENT_PERSONAS)}): first 10 = {CLIENT_PERSONAS[:10]} ...")
print(f"Val rows per selected persona (min/mean/max): "
      f"{val_counts[CLIENT_PERSONAS].min()}/{val_counts[CLIENT_PERSONAS].mean():.1f}/{val_counts[CLIENT_PERSONAS].max()}")

train rows: 3,870 | val rows: 714
Subset 'all_subset': 150 personas
Client personas (150): first 10 = [6, 9, 14, 18, 33, 41, 51, 56, 57, 73] ...
Val rows per selected persona (min/mean/max): 4/4.8/8


In [4]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not available. This notebook needs a GPU.\n"
        "In Google Colab: Runtime → Change runtime type → Hardware accelerator → GPU "
        "(e.g. T4), then Runtime → Restart session and re-run from the top."
    )
print("GPU:", torch.cuda.get_device_name(0))
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    get_peft_model_state_dict,
    prepare_model_for_kbit_training,
    set_peft_model_state_dict,
)
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-0.6B"
MAX_SEQ_LEN = 4096
MAX_SNIPPET_TOKENS = 2048
MAX_ANSWER_TOKENS = 512
MAX_NEW_TOKENS = 512

START_MODE = "hf"
CONTINUE_FROM_EPOCH = 3
CONTINUE_EPOCHS = 1
GLOBAL_EPOCHS = 3
GLOBAL_LR = 1e-3
GLOBAL_WEIGHT_DECAY = 0.0
MAX_GRAD_NORM = 1.0
LOCAL_BATCH_SIZE = 4

LOCAL_LR = 1e-3

ORPO_LAMBDA = 0.2
EVAL_BATCH_SIZE = 32
EVAL_TRAIN = False

model_tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if model_tok.pad_token is None:
    model_tok.pad_token = model_tok.eos_token
model_tok.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

KEEP_COLS = ["user_query", "correct_answer", "incorrect_answers", "related_conversation_snippet"]

def build_system_prompt():
    return (
        "You are a personalised assistant. Use the conversation snippet to find "
        "information or connections relevant to the question, then provide the answer."
    )

def truncate_snippet(snippet, max_tokens):
    ids = model_tok(snippet, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return snippet
    return model_tok.decode(ids[-max_tokens:], skip_special_tokens=True)

def build_user_content(row):
    snippet = truncate_snippet(get_snippet(row), MAX_SNIPPET_TOKENS)
    return (
        "Relevant snippet from our earlier conversation:\n"
        f"{snippet}\n\n"
        f"{parse_user_query(row['user_query'])}"
    )

def build_prompt_text(system_prompt, row):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": build_user_content(row)},
    ]
    return model_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

client_data: Dict[Any, Dict[str, Any]] = {}
for pid in CLIENT_PERSONAS:
    p_train = train_df[train_df["persona_id"] == pid].reset_index(drop=True)
    system_prompt = build_system_prompt()
    rows = p_train[KEEP_COLS].to_dict("records")
    valid_rows = [r for r in rows if parse_incorrect_answers(r.get("incorrect_answers"))]
    client_data[pid] = {
        "rows": rows,
        "valid_rows": valid_rows,
        "n": len(rows),
        "system_prompt": system_prompt,
    }
    print(f"  client persona {pid}: {len(rows)} train examples | {len(valid_rows)} preference pairs")

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

  client persona 6: 21 train examples | 21 preference pairs
  client persona 9: 27 train examples | 27 preference pairs
  client persona 14: 27 train examples | 27 preference pairs
  client persona 18: 22 train examples | 22 preference pairs
  client persona 33: 25 train examples | 25 preference pairs
  client persona 41: 26 train examples | 26 preference pairs
  client persona 51: 23 train examples | 23 preference pairs
  client persona 56: 27 train examples | 27 preference pairs
  client persona 57: 33 train examples | 33 preference pairs
  client persona 73: 28 train examples | 28 preference pairs
  client persona 80: 22 train examples | 22 preference pairs
  client persona 83: 33 train examples | 33 preference pairs
  client persona 87: 23 train examples | 23 preference pairs
  client persona 91: 28 train examples | 28 preference pairs
  client persona 93: 24 train examples | 24 preference pairs
  client persona 94: 32 train examples | 32 preference pairs
  client persona 95: 30 tr

In [5]:
import torch.nn.functional as F

def load_continual_adapter(from_epoch):
    """Continual learning: load base model + a previously saved LoRA adapter (trainable)."""
    adapter_dir = RESULTS_DIR / "global" / f"adapter_epoch_{from_epoch}"
    if not adapter_dir.exists():
        raise FileNotFoundError(f"Adapter to continue from not found: {adapter_dir}")
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto",
        trust_remote_code=True, attn_implementation="sdpa",
    )
    base.config.use_cache = False
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base.enable_input_require_grads()
    m = PeftModel.from_pretrained(base, str(adapter_dir), is_trainable=True)
    m.print_trainable_parameters()
    return m

def load_base_with_adapter():
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto",
        trust_remote_code=True, attn_implementation="sdpa",
    )
    base.config.use_cache = False
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base.enable_input_require_grads()
    m = get_peft_model(base, lora_config)
    return m

def clone_state(model):
    return {k: v.detach().cpu().clone() for k, v in get_peft_model_state_dict(model).items()}

def _build_input_labels(prompt_text, answer_text):
    eos = model_tok.eos_token or ""
    prompt_ids = model_tok(prompt_text, add_special_tokens=False)["input_ids"]
    answer_ids = model_tok(
        str(answer_text) + eos,
        add_special_tokens=False, truncation=True, max_length=MAX_ANSWER_TOKENS,
    )["input_ids"]
    input_ids = (prompt_ids + answer_ids)[:MAX_SEQ_LEN]
    prompt_len = min(len(prompt_ids), MAX_SEQ_LEN)
    labels = ([-100] * prompt_len + input_ids[prompt_len:])[:MAX_SEQ_LEN]
    attention_mask = [1] * len(input_ids)
    return input_ids, attention_mask, labels

def _forward_logits(model, input_ids, attention_mask):
    """Autocast on CUDA; plain forward on CPU (should not happen after CUDA check)."""
    device_type = next(model.parameters()).device.type
    if device_type == "cuda":
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            return model(input_ids=input_ids, attention_mask=attention_mask).logits
    return model(input_ids=input_ids, attention_mask=attention_mask).logits

def sequence_logprob(model, prompt_text, answer_text):
    """Return (sum_logprob, num_tokens) over the answer tokens."""
    input_ids, attention_mask, labels = _build_input_labels(prompt_text, answer_text)
    device = next(model.parameters()).device
    ids = torch.tensor([input_ids], device=device)
    attn = torch.tensor([attention_mask], device=device)
    lab = torch.tensor([labels], device=device)
    logits = _forward_logits(model, ids, attn)
    shift_logits = logits[:, :-1, :].float()
    shift_labels = lab[:, 1:]
    valid = shift_labels != -100
    safe = shift_labels.masked_fill(~valid, 0)
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_logps = log_probs.gather(dim=-1, index=safe.unsqueeze(-1)).squeeze(-1)
    total = (token_logps * valid).sum()
    count = valid.sum().clamp(min=1)
    return total, count

def _log1mexp(x):
    """Numerically stable log(1 - exp(x)) for x < 0."""
    return torch.log(-torch.expm1(x.clamp(max=-1e-6)))

def orpo_loss_for_row(model, system_prompt, row):
    prompt = build_prompt_text(system_prompt, row)
    chosen = str(row["correct_answer"])
    rejected = parse_incorrect_answers(row["incorrect_answers"])[0]

    sum_c, n_c = sequence_logprob(model, prompt, chosen)
    sum_r, n_r = sequence_logprob(model, prompt, rejected)
    mean_c = sum_c / n_c
    mean_r = sum_r / n_r

    nll = -mean_c

    log_odds = (mean_c - _log1mexp(mean_c)) - (mean_r - _log1mexp(mean_r))
    or_loss = -F.logsigmoid(log_odds)

    loss = nll + ORPO_LAMBDA * or_loss
    parts = {
        "loss": float(loss.detach().cpu()),
        "nll": float(nll.detach().cpu()),
        "or_loss": float(or_loss.detach().cpu()),
    }
    return loss, parts

def local_orpo_train(model, rows, system_prompt, epochs, lr, batch_size=LOCAL_BATCH_SIZE):
    """One client's local ORPO on its preference pairs."""
    model.train()
    model.config.use_cache = False
    valid_rows = [r for r in rows if parse_incorrect_answers(r.get("incorrect_answers"))]
    if not valid_rows:
        print("  warning: no preference pairs for local ORPO")
        return {"mean_loss": None, "epoch_losses": [], "n_batches": 0}

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr)
    n = len(valid_rows)
    indices = list(range(n))
    epoch_losses: List[float] = []
    n_batches = 0

    for epoch in range(epochs):
        random.shuffle(indices)
        batch_losses: List[float] = []
        for start in tqdm(range(0, n, batch_size), desc=f"Local epoch {epoch + 1}/{epochs}", leave=False):
            batch_rows = [valid_rows[i] for i in indices[start:start + batch_size]]
            optimizer.zero_grad()
            losses = []
            for row in batch_rows:
                loss, _ = orpo_loss_for_row(model, system_prompt, row)
                losses.append(loss)
            if losses:
                batch_loss = torch.stack(losses).mean()
                batch_loss.backward()
                optimizer.step()
                batch_losses.append(float(batch_loss.detach().cpu()))
                n_batches += 1
        if batch_losses:
            epoch_losses.append(sum(batch_losses) / len(batch_losses))

    del optimizer
    gc.collect()
    torch.cuda.empty_cache()
    mean_loss = sum(epoch_losses) / len(epoch_losses) if epoch_losses else None
    return {"mean_loss": mean_loss, "epoch_losses": epoch_losses, "n_batches": n_batches}

class ClientBatchSampler:
    """Yields fixed-size batches of a client's preference pairs, reshuffling on wrap-around."""

    def __init__(self, rows, batch_size, seed=SEED):
        self.rows = rows
        self.batch_size = batch_size
        self.rng = random.Random(seed)
        self.order: List[int] = []
        self.pos = 0
        self._reshuffle()

    def _reshuffle(self):
        self.order = list(range(len(self.rows)))
        self.rng.shuffle(self.order)
        self.pos = 0

    def new_epoch(self):
        """Start a fresh pass over this client's data (reshuffle, reset pointer)."""
        self._reshuffle()

    def next_batch(self):
        if not self.rows:
            return []
        batch = []
        while len(batch) < self.batch_size:
            if self.pos >= len(self.order):
                self._reshuffle()
            batch.append(self.rows[self.order[self.pos]])
            self.pos += 1
        return batch

def orpo_batch_loss(model, system_prompt, batch_rows):
    """Mean ORPO loss over a batch of preference pairs (keeps the graph for backward)."""
    losses, parts_acc = [], {"nll": 0.0, "or_loss": 0.0}
    for row in batch_rows:
        loss, parts = orpo_loss_for_row(model, system_prompt, row)
        losses.append(loss)
        parts_acc["nll"] += parts["nll"]
        parts_acc["or_loss"] += parts["or_loss"]
    batch_loss = torch.stack(losses).mean()
    n = max(1, len(losses))
    return batch_loss, {"nll": parts_acc["nll"] / n, "or_loss": parts_acc["or_loss"] / n}

In [6]:
import math

if START_MODE == "adapter":
    model = load_continual_adapter(CONTINUE_FROM_EPOCH)
    _start_epoch = CONTINUE_FROM_EPOCH + 1
    _end_epoch = CONTINUE_FROM_EPOCH + CONTINUE_EPOCHS
    print(f"Continual learning from adapter_epoch_{CONTINUE_FROM_EPOCH}: "
          f"training epochs {_start_epoch}..{_end_epoch}")
else:
    model = load_base_with_adapter()
    _start_epoch = 1
    _end_epoch = GLOBAL_EPOCHS
    print(f"Fresh HuggingFace base model: training epochs {_start_epoch}..{_end_epoch}")

model.train()
model.config.use_cache = False

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=GLOBAL_LR, weight_decay=GLOBAL_WEIGHT_DECAY)

active_personas = [pid for pid in CLIENT_PERSONAS if client_data[pid]["valid_rows"]]
num_active = len(active_personas)
samplers = {pid: ClientBatchSampler(client_data[pid]["valid_rows"], LOCAL_BATCH_SIZE)
            for pid in active_personas}

steps_per_epoch = max(math.ceil(len(client_data[pid]["valid_rows"]) / LOCAL_BATCH_SIZE)
                      for pid in active_personas)
_num_train_epochs = _end_epoch - _start_epoch + 1
GLOBAL_STEPS = _num_train_epochs * steps_per_epoch
print(f"Active clients: {num_active}/{len(CLIENT_PERSONAS)} | steps/epoch: {steps_per_epoch} "
      f"| global steps: {GLOBAL_STEPS}")

GLOBAL_DIR = RESULTS_DIR / "global"
GLOBAL_DIR.mkdir(parents=True, exist_ok=True)
epoch_adapter_dirs: List[str] = []
training_log: Dict[str, Any] = {"method": "federated_orpo_gradavg_global", "steps": []}

global_state = clone_state(model)
global_step = 0
for epoch in range(_start_epoch, _end_epoch + 1):
    for s in samplers.values():
        s.new_epoch()
    epoch_step_losses: List[float] = []

    for step in tqdm(range(1, steps_per_epoch + 1), desc=f"Epoch {epoch}/{_end_epoch}"):
        set_peft_model_state_dict(model, global_state)

        optimizer.zero_grad(set_to_none=True)
        client_step_losses: List[float] = []
        for pid in active_personas:
            batch = samplers[pid].next_batch()
            batch_loss, _ = orpo_batch_loss(model, client_data[pid]["system_prompt"], batch)
            (batch_loss / num_active).backward()
            client_step_losses.append(float(batch_loss.detach().cpu()))
        if MAX_GRAD_NORM:
            torch.nn.utils.clip_grad_norm_(params, MAX_GRAD_NORM)
        optimizer.step()

        global_state = clone_state(model)
        global_step += 1
        mean_step_loss = sum(client_step_losses) / len(client_step_losses)
        epoch_step_losses.append(mean_step_loss)
        training_log["steps"].append({
            "epoch": epoch, "step": step, "global_step": global_step,
            "mean_client_loss": mean_step_loss,
        })

    epoch_dir = GLOBAL_DIR / f"adapter_epoch_{epoch}"
    epoch_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(epoch_dir))
    model_tok.save_pretrained(str(epoch_dir))
    epoch_adapter_dirs.append(str(epoch_dir))
    print(f"epoch {epoch}/{_end_epoch} | updates this epoch={steps_per_epoch} "
          f"(global_step={global_step}) | mean_loss={sum(epoch_step_losses) / len(epoch_step_losses):.4f} "
          f"| saved {epoch_dir.name}")

global_state = clone_state(model)

with open(GLOBAL_DIR / "training_losses.json", "w", encoding="utf-8") as f:
    json.dump(training_log, f, indent=2)

global_adapter_dir = GLOBAL_DIR / "adapter"
model.save_pretrained(str(global_adapter_dir))
model_tok.save_pretrained(str(global_adapter_dir))

with open(GLOBAL_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump({
        "method": "federated_orpo_gradavg_global",
        "aggregation": "gradient_averaging_fedsgd",
        "single_global_optimizer": True,
        "subset": SUBSET_NAME,
        "client_personas": [int(p) for p in CLIENT_PERSONAS],
        "num_clients": len(CLIENT_PERSONAS),
        "num_active_clients": num_active,
        "min_val_rows": MIN_VAL_ROWS,
                "seed": SEED,
        "start_mode": START_MODE,
        "continue_from_epoch": CONTINUE_FROM_EPOCH if START_MODE == "adapter" else None,
        "continue_epochs": CONTINUE_EPOCHS if START_MODE == "adapter" else None,
        "trained_epoch_range": [int(_start_epoch), int(_end_epoch)],
        "global_epochs": GLOBAL_EPOCHS,
        "steps_per_epoch": steps_per_epoch,
        "global_steps": GLOBAL_STEPS,
        "global_lr": GLOBAL_LR,
        "global_weight_decay": GLOBAL_WEIGHT_DECAY,
        "max_grad_norm": MAX_GRAD_NORM,
        "local_batch_size": LOCAL_BATCH_SIZE,
        "orpo_lambda": ORPO_LAMBDA,
        "epoch_adapter_dirs": epoch_adapter_dirs,
        "final_adapter_dir": str(global_adapter_dir),
    }, f, indent=2)

del optimizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Saved global federated ORPO (gradient-averaging) adapter to:", global_adapter_dir.resolve())

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Fresh HuggingFace base model: training epochs 1..3
Active clients: 150/150 | steps/epoch: 10 | global steps: 30


...

epoch 1/3 | updates this epoch=10 (global_step=10) | mean_loss=2.8797 | saved adapter_epoch_1


Epoch 2/3:   0%|          | 0/10 [00:00<?, ?it/s]

epoch 2/3 | updates this epoch=10 (global_step=20) | mean_loss=2.3384 | saved adapter_epoch_2


Epoch 3/3:   0%|          | 0/10 [00:00<?, ?it/s]

epoch 3/3 | updates this epoch=10 (global_step=30) | mean_loss=1.9913 | saved adapter_epoch_3
Saved global federated ORPO (gradient-averaging) adapter to: /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_federated_orpo_gradavg_snippet/global/adapter


In [ ]:
def load_global_model(epoch=None):
    """Load the federated global ORPO LoRA adapter. Use epoch=1..FED_ROUNDS for round checkpoints."""
    if epoch is not None:
        adapter_dir = RESULTS_DIR / "global" / f"adapter_epoch_{epoch}"
    else:
        adapter_dir = RESULTS_DIR / "global" / "adapter"
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto",
        trust_remote_code=True, attn_implementation="sdpa",
    )
    base.config.use_cache = True
    m = PeftModel.from_pretrained(base, str(adapter_dir))
    m.eval()
    tok = AutoTokenizer.from_pretrained(str(adapter_dir))
    return m, tok

print("Final global adapter:", RESULTS_DIR / "global" / "adapter")
print("Per-round checkpoints:", RESULTS_DIR / "global" / "adapter_epoch_<n>")
print("Per-persona val preds:", RESULTS_DIR / "persona_<id>" / "val_predictions.csv")